**Descripción del problema**

El problema de la mochila (Knapsack Problem) es un problema de optimización combinatoria.
Dada una mochila con una capacidad máxima de peso W , y un conjunto de n objetos, cada uno
con un peso wi y un valor vi , el objetivo es seleccionar un subconjunto de objetos que maximice el
valor total sin exceder la capacidad máxima de la mochila.
Este problema tiene una amplia aplicabilidad en diferentes contextos del mundo real. Por ejem-
plo:

- Selección de proyectos en una empresa con un presupuesto limitado.
- Asignación de espacio en contenedores de carga para maximizar el valor transportado.
- Planificación de envı́os en empresas de mensajerı́a.
- Optimización de préstamos o créditos.
- Selección de caracterı́sticas para mejorar la precisión de modelos predictivos.
- Entre muchos otros.

In [2]:
"""
    - Importaciones necesarias.
"""
import numpy as np
import numpy.typing as npt
import random
import matplotlib.pyplot as plt # Poder crear graficos
from matplotlib.ticker import PercentFormatter # Crear porcentajes.

In [ ]:
"""
    - Generar cargar de manera aleatoria.
"""

def generar_carga():
    """
    Genera una población aleatoria.

    Retorna
    -------
    npt.NDArray[np.int8]
        Matriz de numpy de forma (100, 30)
        - 100 cargas
        - 30 genes por individuo
        - valores entre 0 y 1 donde 1 es carga y 0 es no tener carga.
    """
    # TODO: Generar una población aleatoria conformada por individuos que contienen 8 genes.
    # Cada dimensión representa la columna de la posición de la reina, se tiene como máximo 8 columnas.
    # El valor del gen representa la fila donde está colocada la reina, se tiene los valores 0-7,
    # considerando el cero como la primera fila.

    poblacion: npt.NDArray[np.int8] = np.random.randint(0,2,(100,30))
    return poblacion

In [22]:
"""
    - Esta funciion sirve para checar los datos de las diferentes cargas
    - Los datos de nuestro objetos son:
        • Primer campo: Nombre del objeto.
        • Segundo campo: Valor del objeto.
        • Tercer campo: Peso del objeto.
"""
def datos_carga(posicion:int)->tuple[str, int, int]:

    objetos:list[tuple[str, int, int]] = [
                ('Botella de agua', 30, 6),
                ('Comida deshidratada', 40, 8),
                ('Botiquı́n de primeros auxilios', 50, 10),
                ('Linterna LED', 28, 4),
                ('Baterı́as recargables', 14, 2),
                ('Cuchillo multiusos', 30, 5),
                ('Mapa topográfico', 15, 3),
                ('Brújula profesional', 16, 3),
                ('Tienda de campaña ultraligera', 70, 14),
                ('Saco de dormir térmico', 65, 13),
                ('Colchoneta inflable', 30, 6),
                ('Chamarra impermeable', 40, 9),
                ('Guantes térmicos', 15, 3),
                ('Filtro portátil de agua', 35, 5),
                ('Cuerda de escalada', 50, 12),
                ('Mosquetones de acero', 22, 5),
                ('Estufa portátil', 36, 8),
                ('Cartucho de gas', 20, 4),
                ('Kit de reparación', 18, 4),
                ('Radio de emergencia', 45, 9),
                ('Panel solar portátil', 65, 12),
                ('GPS de montaña', 42, 6),
                ('Cámara compacta', 40, 6),
                ('Cuaderno impermeable', 10, 2),
                ('Teléfono satelital', 40, 5),
                ('Laptop ultraligera', 110, 15),
                ('Power bank alta capacidad', 28, 4),
                ('Raciones energéticas', 28, 5),
                ('Manta térmica', 12, 2),
                ('Silbato de emergencia', 10, 2)
            ]
    return objetos[posicion]

In [ ]:
"""
    - Calculamos su aptitud (fitness) de cada carga. Maximizar el valor total con la restricción máxima de peso W
        - Fitness-Cero
        - Lineal
        - Cuadrática
        - Exponencial

"""
from abc import ABC, abstractmethod

class Penalizacion(ABC):

    def __init__(self, peso_maximo:int=50):
        self.peso_maximo = peso_maximo
        self.peso_total = 0

    def calcular_peso_total(self,carga: npt.NDArray[np.int8]) -> None:
        self.peso_total = 0

        for posicion, objeto in enumerate(carga):
            if objeto == 1:
                nombre,valor,peso = datos_carga(posicion=posicion)
                self.peso_total +=  peso

    @abstractmethod
    def calcular_aptitud(self)->int:
        pass


class Fitness_Cero(Penalizacion):

    def calcular_aptitud(self) -> int:
        if self.peso_total > self.peso_maximo: 
            return 0
        
        return self.peso_total 
                
class Lineal(Penalizacion):

    def calcular_aptitud(self)-> int:
        
        alfa:int = 1
        valor_penalizacion: int = 0
        exceso: int = 0
        
        if self.peso_total <= self.peso_maximo:
            return 0

        exceso = (self.peso_total - self.peso_maximo)
        valor_penalizacion = (alfa * exceso)
        
        return  valor_penalizacion
    
class Cuadratica(Penalizacion):
    import math

    def calcular_aptitud(self)-> int:
        
        alfa:int = 0.2
        valor_penalizacion: int = 0
        exceso:int = 0

        if self.peso_total <= self.peso_total:
            return 0

        exceso = (self.peso_total-self.peso_maximo)

        valor_penalizacion = (alfa * (self.math.pow(exceso,2)))
        
        return  valor_penalizacion
    
class Exponencial(Penalizacion):
    import math
    
    def calcular_aptitud(self) -> int:
        alfa:int = 1
        valor_penalizacion: int = 0
        exceso: int = 0

        if self.peso_total <= self.peso_total:
            return 0

        exceso = self.peso_total-self.peso_maximo

        valor_penalizacion = (alfa * (self.math.exp(exceso)))
        
        return  valor_penalizacion

In [ ]:
"""
    - Calculo del fitgess de una carga: El fitness final está definido por: 
            - fitness = valor total - penalización
"""

class Fitness():
    
    metodos_penalizacion: dict[str,Penalizacion] = {
            "fitnes_cero": Fitness_Cero,
            "lineal": Lineal,
            "cuadratica":Cuadratica,
            "exponencial":Exponencial,
        }

    def __init__(self,carga:npt.NDArray[np.int8]):
        self.carga:npt.NDArray[np.int8] = carga
        self.valor_total: int = 0
    
    def calcular_valor_total(self)-> None:
         self.valor_total = 0
         for posicion, objeto in enumerate(self.carga):
            if objeto == 1:
                nombre, valor , peso = datos_carga(posicion=posicion)
                self.valor_total += valor
    
    def calcular_fitness(self,penalizacion_selecionada:str = "fitnes_cero")-> int:
        self.calcular_valor_total()
        penalizacion: Penalizacion = self.metodos_penalizacion[penalizacion_selecionada]()
        penalizacion.calcular_peso_total(carga=self.carga)
        valor_penalizacion: int = penalizacion.calcular_aptitud()
        
        return self.valor_total - valor_penalizacion
    

In [63]:
carga = datos_carga(posicion=1)
calcular_fitnes = Fitness(carga)

print(calcular_fitnes.calcular_fitness(penalizacion_selecionada="lineal"))

0
